In [11]:
import sys
from pathlib import Path

# Add the project root to Python path
project_root = Path().absolute().parent.parent.parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
from src.app.vectorstores.search_factory import SearchServiceFactory
from src.app.core.config import settings
from src.app.vectorstores.sample_data import get_sample_documents

Project root: /Users/ahmedashraf/Documents/GitHub/PhysioAssist-AI


In [12]:
search_service = SearchServiceFactory.create_elasticsearch()
info = search_service.client.info()

print("Connection successful!")
print(f"Cluster name: {info['cluster_name']}")
print(f"ElasticSearch version: {info['version']['number']}")
print("✓ ElasticSearch Service created successfully")


Connection successful!
Cluster name: docker-cluster
ElasticSearch version: 9.2.3
✓ ElasticSearch Service created successfully


In [13]:
sample_docs = get_sample_documents()
print(f"Loaded {len(sample_docs)} documents")

Loaded 10 documents


In [14]:
documents_with_embeddings = []
for i, doc in enumerate(sample_docs, 1):
    print(f"  Processing {i}/{len(sample_docs)}: {doc['title'][:50]}...")
    embedding = search_service.embed(doc['content'])
    doc_with_embedding = doc.copy()
    doc_with_embedding['embedding'] = embedding
    documents_with_embeddings.append(doc_with_embedding)

  Processing 1/10: Text Neck Syndrome...
  Processing 2/10: Whiplash Injury...
  Processing 3/10: Neck Muscle Strain...
  Processing 4/10: Cervical Radiculopathy...
  Processing 5/10: Cervical Disc Herniation...
  Processing 6/10: Cervical Spondylosis...
  Processing 7/10: Shoulder Impingement Syndrome...
  Processing 8/10: Rotator Cuff Tear...
  Processing 9/10: ACL Tear...
  Processing 10/10: PCL Tear...


In [15]:
# Check actual embedding dimensions
test_embedding = search_service.embed("test text")
print(f"Actual embedding dimensions: {len(test_embedding)}")
print(f"Embedding provider: {search_service.embedder.__class__.__name__}")
print(f"Embedding model: {search_service.embedder.model}")
print(f"First 5 values: {test_embedding[:5]}")

Actual embedding dimensions: 1536
Embedding provider: OpenAIEmbeddingProvider
Embedding model: text-embedding-3-small
First 5 values: [-0.025677789002656937, 0.00506477989256382, 0.04459746181964874, -0.06313661485910416, -0.03208582103252411]


In [16]:
if search_service.index_document(settings.INDEX_NAME, documents_with_embeddings[0]):
    print("Single document indexed successfully")

Single document indexed successfully


In [17]:
remaining_docs = documents_with_embeddings[1:]
success = search_service.bulk_index(settings.INDEX_NAME, remaining_docs)
print(f"✓ Bulk indexing completed: {success}")

Successfully indexed 9 documents
✓ Bulk indexing completed: True


In [18]:
count = search_service.get_document_count(settings.INDEX_NAME)
print(f"\nTotal documents in index: {count}")


Total documents in index: 10


In [9]:
# search_service.delete_index(INDEX_NAME)